# Floyd-Warshall and the All-Pairs Shortest Path Problem

This notebook is meant to be run live in class, either in Jupyter or in Google Colab.

Goals for today:

* recall the single-source shortest path algorithms already covered in the course: Dijkstra, A*, and Bellman-Ford
* motivate why all-pairs shortest paths (APSP) is a different problem
* build a correct Floyd-Warshall implementation with path reconstruction
* visualize the distance matrix as the algorithm progresses
* verify our result against a trusted library implementation
* finish with a short survey of other shortest path algorithms not covered in the course


## Quick Recap: Single-Source Shortest Paths Already Covered

Earlier in the course we studied algorithms that answer shortest-path questions **from one chosen source vertex**.

| Algorithm | Main idea | Negative edges allowed? | Typical use |
| --- | --- | --- | --- |
| Dijkstra | Greedy expansion from the current closest vertex | No | Fast single-source shortest paths in graphs with non-negative weights |
| A* | Dijkstra plus a heuristic toward one target | No, unless the heuristic setting is special and edge weights remain compatible | Goal-directed search when a good heuristic exists |
| Bellman-Ford | Repeated edge relaxation | Yes | Single-source shortest paths when negative edges may appear |

All three are **single-source** algorithms.

If we need answers for **every pair of vertices**, then rerunning a single-source algorithm from every source is one option, but not always the cleanest or most informative one.


## Why Do We Need All-Pairs Shortest Paths?

APSP is useful when we need the whole distance table, not just one row of it.

Typical situations:

* we want the travel cost between every pair of cities, routers, or campuses
* we want to answer many shortest-path queries on the same graph
* we want graph-wide quantities such as a diameter-like maximum finite shortest-path distance
* we want a matrix view of reachability and path cost, not only one source at a time

Important warning:

* Floyd-Warshall works with negative edges
* Floyd-Warshall does **not** give meaningful shortest paths when a reachable negative cycle exists
* if a negative cycle exists, we can keep going around that cycle and reduce the path cost without bound


In [ ]:
import importlib.util
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, FancyArrowPatch

NETWORKX_AVAILABLE = importlib.util.find_spec("networkx") is not None
if NETWORKX_AVAILABLE:
    import networkx as nx
else:
    nx = None

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = False

print(f"Python version: {sys.version.split()[0]}")
if NETWORKX_AVAILABLE:
    print(f"NetworkX version: {nx.__version__}")
else:
    print("NetworkX is not installed in this runtime. Graph drawing still works, but library verification will be skipped.")


## Floyd-Warshall as Dynamic Programming

Number the vertices as `0, 1, ..., V-1`.

Let `d_k(i, j)` mean:

* the shortest path distance from `i` to `j`
* using only vertices from the set `{0, 1, ..., k-1}` as intermediate vertices

Then the recurrence is:

$d_{k+1}(i, j) = \min\left(d_k(i, j), d_k(i, k) + d_k(k, j)\right)$

At each stage we decide whether allowing one more intermediate vertex improves the best known path.

That gives the classic complexity:

* time: `O(V^3)`
* space: `O(V^2)`


In [ ]:
def adjacency_matrix_from_edges(labels, edges):
    index = {label: i for i, label in enumerate(labels)}
    n = len(labels)
    matrix = [[0 if i == j else math.inf for j in range(n)] for i in range(n)]
    for u, v, w in edges:
        matrix[index[u]][index[v]] = min(matrix[index[u]][index[v]], w)
    return matrix


def format_distance(value):
    if value == math.inf:
        return "INF"
    if abs(value - round(value)) < 1e-9:
        return str(int(round(value)))
    return f"{value:.2f}"


def print_distance_matrix(dist, labels):
    width = max(5, max(len(str(label)) for label in labels) + 1)
    print(" " * width, end="")
    for label in labels:
        print(f"{label:>{width}}", end="")
    print()
    for label, row in zip(labels, dist):
        print(f"{label:>{width}}", end="")
        for value in row:
            print(f"{format_distance(value):>{width}}", end="")
        print()


def floyd_warshall_with_path(weight_matrix, labels=None, record_steps=False):
    n = len(weight_matrix)
    labels = list(range(n)) if labels is None else list(labels)
    dist = [row[:] for row in weight_matrix]
    nxt = [[None] * n for _ in range(n)]

    for i in range(n):
        dist[i][i] = min(dist[i][i], 0)
        for j in range(n):
            if i != j and dist[i][j] != math.inf:
                nxt[i][j] = j

    snapshots = []
    if record_steps:
        snapshots.append({
            "k": None,
            "label": "start",
            "dist": [row[:] for row in dist],
            "updates": [],
        })

    for k in range(n):
        updates = []
        for i in range(n):
            if dist[i][k] == math.inf:
                continue
            for j in range(n):
                if dist[k][j] == math.inf:
                    continue
                candidate = dist[i][k] + dist[k][j]
                if candidate < dist[i][j]:
                    old = dist[i][j]
                    dist[i][j] = candidate
                    nxt[i][j] = nxt[i][k]
                    updates.append((labels[i], labels[j], labels[k], old, candidate))
        if record_steps:
            snapshots.append({
                "k": k,
                "label": labels[k],
                "dist": [row[:] for row in dist],
                "updates": updates,
            })

    negative_cycle_vertices = [labels[v] for v in range(n) if dist[v][v] < 0]

    return {
        "labels": labels,
        "dist": dist,
        "next": nxt,
        "snapshots": snapshots,
        "negative_cycle_vertices": negative_cycle_vertices,
    }


def reconstruct_path(start, end, nxt, labels):
    label_to_index = {label: i for i, label in enumerate(labels)}
    i = label_to_index[start]
    j = label_to_index[end]

    if start == end:
        return [start]
    if nxt[i][j] is None:
        return None

    path = [labels[i]]
    steps = 0
    while i != j:
        i = nxt[i][j]
        if i is None:
            return None
        path.append(labels[i])
        steps += 1
        if steps > len(labels):
            raise ValueError("Path reconstruction failed; a negative cycle may affect this pair.")
    return path


In [ ]:
def digraph_from_matrix(matrix, labels):
    if not NETWORKX_AVAILABLE:
        raise RuntimeError("networkx is not available in this runtime")
    graph = nx.DiGraph()
    graph.add_nodes_from(labels)
    for i, u in enumerate(labels):
        for j, v in enumerate(labels):
            if i != j and matrix[i][j] != math.inf:
                graph.add_edge(u, v, weight=matrix[i][j])
    return graph


def circular_positions(labels, radius=1.0):
    angles = np.linspace(math.pi / 2, math.pi / 2 - 2 * math.pi, len(labels), endpoint=False)
    return {
        label: (radius * math.cos(angle), radius * math.sin(angle))
        for label, angle in zip(labels, angles)
    }


def draw_weighted_digraph(matrix, labels, path=None, title=""):
    pos = circular_positions(labels)
    path_edges = set(zip(path, path[1:])) if path and len(path) > 1 else set()
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect("equal")

    for i, u in enumerate(labels):
        x1, y1 = pos[u]
        for j, v in enumerate(labels):
            if i == j or matrix[i][j] == math.inf:
                continue

            x2, y2 = pos[v]
            dx = x2 - x1
            dy = y2 - y1
            length = math.hypot(dx, dy)
            shrink = 0.16
            start = (x1 + shrink * dx / length, y1 + shrink * dy / length)
            end = (x2 - shrink * dx / length, y2 - shrink * dy / length)

            reverse_exists = matrix[j][i] != math.inf
            if reverse_exists:
                rad = 0.18 if i < j else -0.18
            else:
                rad = 0.0

            edge = (u, v)
            color = "#dc2626" if edge in path_edges else "#6b7280"
            width = 2.8 if edge in path_edges else 1.4
            arrow = FancyArrowPatch(
                start,
                end,
                arrowstyle="-|>",
                mutation_scale=16,
                linewidth=width,
                color=color,
                connectionstyle=f"arc3,rad={rad}",
            )
            ax.add_patch(arrow)

            mid_x = (start[0] + end[0]) / 2
            mid_y = (start[1] + end[1]) / 2
            if rad != 0.0:
                offset_x = -0.12 * dy
                offset_y = 0.12 * dx
                if rad < 0:
                    offset_x *= -1
                    offset_y *= -1
                mid_x += offset_x
                mid_y += offset_y

            ax.text(
                mid_x,
                mid_y,
                format_distance(matrix[i][j]),
                fontsize=10,
                ha="center",
                va="center",
                bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "none", "alpha": 0.9},
            )

    for label in labels:
        x, y = pos[label]
        facecolor = "#fca5a5" if path and label in path else "#93c5fd"
        circle = Circle((x, y), radius=0.14, facecolor=facecolor, edgecolor="#1f2937", linewidth=1.2)
        ax.add_patch(circle)
        ax.text(x, y, label, ha="center", va="center", fontsize=11, fontweight="bold")

    plt.title(title)
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.35, 1.35)
    ax.axis("off")
    plt.show()


def plot_distance_heatmap(dist, labels, title="Distance matrix"):
    array = np.array(
        [[np.nan if value == math.inf else float(value) for value in row] for row in dist],
        dtype=float,
    )
    cmap = plt.cm.YlGnBu.copy()
    cmap.set_bad(color="#e5e7eb")

    plt.figure(figsize=(6, 5))
    plt.imshow(array, cmap=cmap)
    plt.colorbar(label="distance")
    plt.xticks(range(len(labels)), labels)
    plt.yticks(range(len(labels)), labels)
    plt.xlabel("destination")
    plt.ylabel("source")
    plt.title(title)

    for i in range(len(labels)):
        for j in range(len(labels)):
            plt.text(j, i, format_distance(dist[i][j]), ha="center", va="center", color="black")

    plt.tight_layout()
    plt.show()


def show_snapshot(result, step):
    snapshot = result["snapshots"][step]
    labels = result["labels"]
    if snapshot["k"] is None:
        title = "Initial distance matrix"
    else:
        title = f"After allowing {snapshot['label']} as an intermediate vertex"

    plot_distance_heatmap(snapshot["dist"], labels, title=title)

    if snapshot["updates"]:
        print(f"Updated entries in this step: {len(snapshot['updates'])}")
        for u, v, k, old, new in snapshot["updates"]:
            print(f"{u} -> {v}: {format_distance(old)} became {format_distance(new)} using {k}")
    else:
        print("No entries changed in this step.")


## Example 1: Positive Edge Weights

We start with a small directed graph that has only non-negative edge weights.

This is a good warm-up because:

* the answers are easier to sanity-check by hand
* we can watch the distance matrix improve step by step
* later we can verify the whole result against NetworkX


In [ ]:
labels1 = ["A", "B", "C", "D", "E"]
edges1 = [
    ("A", "B", 3),
    ("A", "C", 10),
    ("A", "E", 7),
    ("B", "C", 2),
    ("B", "D", 6),
    ("C", "D", 1),
    ("C", "E", 2),
    ("D", "E", 2),
    ("E", "A", 4),
    ("E", "D", 5),
]

matrix1 = adjacency_matrix_from_edges(labels1, edges1)
draw_weighted_digraph(matrix1, labels1, title="Example 1: directed weighted graph")


In [ ]:
result1 = floyd_warshall_with_path(matrix1, labels1, record_steps=True)

print("All-pairs shortest path distances for Example 1")
print_distance_matrix(result1["dist"], labels1)
plot_distance_heatmap(result1["dist"], labels1, title="Example 1: final shortest-path matrix")

for source, target in [("A", "D"), ("E", "C"), ("B", "E")]:
    path = reconstruct_path(source, target, result1["next"], labels1)
    cost = result1["dist"][labels1.index(source)][labels1.index(target)]
    print(f"{source} -> {target}: cost={format_distance(cost)}, path={path}")

sample_path = reconstruct_path("A", "D", result1["next"], labels1)
draw_weighted_digraph(matrix1, labels1, path=sample_path, title="Example 1: shortest path A -> D highlighted")


In [ ]:
show_snapshot(result1, step=0)
show_snapshot(result1, step=2)

finite_pairs = []
for i, source in enumerate(labels1):
    for j, target in enumerate(labels1):
        if i != j and result1["dist"][i][j] != math.inf:
            finite_pairs.append((source, target, result1["dist"][i][j]))

diameter_pair = max(finite_pairs, key=lambda item: item[2])
print(
    "One APSP-style question we can now answer immediately:",
    f"largest finite shortest-path distance = {format_distance(diameter_pair[2])} for {diameter_pair[0]} -> {diameter_pair[1]}",
)


## Optional Interactive Exploration

If `ipywidgets` is available, the next cell gives a slider for browsing the saved snapshots.

If widgets are not available, the notebook still works fine because `show_snapshot(result, step)` can be called manually.


In [ ]:
try:
    from IPython import get_ipython
    from ipywidgets import IntSlider, interact
    notebook_ui = get_ipython() is not None
except Exception:
    IntSlider = None
    interact = None
    notebook_ui = False

if interact is not None and notebook_ui:
    @interact(step=IntSlider(min=0, max=len(result1["snapshots"]) - 1, step=1, value=0))
    def browse_example_1(step=0):
        show_snapshot(result1, step)
else:
    print("Run this cell in Jupyter or Colab with ipywidgets installed to get the slider.")


## Example 2: Negative Edges but No Negative Cycle

This is where Floyd-Warshall becomes more interesting.

* Dijkstra is not reliable once negative edges appear
* Bellman-Ford is still valid for a single source
* Floyd-Warshall remains valid for the all-pairs problem as long as there is no negative cycle


In [ ]:
labels2 = ["S", "A", "B", "C", "D"]
edges2 = [
    ("S", "A", 4),
    ("S", "B", 5),
    ("A", "B", -2),
    ("A", "C", 6),
    ("B", "C", 3),
    ("B", "D", 8),
    ("C", "A", 1),
    ("C", "D", 2),
    ("D", "S", 7),
]

matrix2 = adjacency_matrix_from_edges(labels2, edges2)
draw_weighted_digraph(matrix2, labels2, title="Example 2: negative edges, but no negative cycle")

result2 = floyd_warshall_with_path(matrix2, labels2, record_steps=False)
print_distance_matrix(result2["dist"], labels2)
plot_distance_heatmap(result2["dist"], labels2, title="Example 2: final shortest-path matrix")

path2 = reconstruct_path("S", "D", result2["next"], labels2)
cost2 = result2["dist"][labels2.index("S")][labels2.index("D")]
print(f"S -> D: cost={format_distance(cost2)}, path={path2}")
draw_weighted_digraph(matrix2, labels2, path=path2, title="Example 2: shortest path S -> D highlighted")
print("Negative cycle detected:", bool(result2["negative_cycle_vertices"]))


## Example 3: Detecting a Negative Cycle

If any diagonal entry becomes negative, then a negative cycle exists.

That means shortest paths are not well-defined for pairs that can keep looping through that cycle.


In [ ]:
labels3 = ["P", "Q", "R", "T"]
edges3 = [
    ("P", "Q", 1),
    ("Q", "R", -2),
    ("R", "P", -2),
    ("P", "T", 4),
    ("R", "T", 2),
]

matrix3 = adjacency_matrix_from_edges(labels3, edges3)
draw_weighted_digraph(matrix3, labels3, title="Example 3: graph containing a negative cycle")

result3 = floyd_warshall_with_path(matrix3, labels3, record_steps=False)
print_distance_matrix(result3["dist"], labels3)
plot_distance_heatmap(result3["dist"], labels3, title="Example 3: diagonal entries reveal a negative cycle")
print("Vertices on a negative cycle:", result3["negative_cycle_vertices"])
if result3["negative_cycle_vertices"]:
    print("Do not trust reconstructed shortest paths for pairs affected by that cycle.")


## Verification Against NetworkX

For classroom code, it is useful to verify our implementation against a trusted library.

If `networkx` is available in the runtime, we use `networkx.floyd_warshall_predecessor_and_distance` to compare distance tables and one reconstructed path.


In [ ]:
def verify_against_networkx(matrix, labels, sample_pair):
    if not NETWORKX_AVAILABLE:
        print("Skipping library verification because networkx is not installed in this runtime.")
        return

    graph = digraph_from_matrix(matrix, labels)
    nx_pred, nx_dist = nx.floyd_warshall_predecessor_and_distance(graph, weight="weight")
    ours = floyd_warshall_with_path(matrix, labels)

    mismatch = None
    for i, source in enumerate(labels):
        for j, target in enumerate(labels):
            expected = nx_dist[source].get(target, math.inf)
            actual = ours["dist"][i][j]
            same = (math.isinf(expected) and math.isinf(actual)) or abs(expected - actual) < 1e-9
            if not same:
                mismatch = (source, target, actual, expected)
                break
        if mismatch is not None:
            break

    print("Distance tables match NetworkX:", mismatch is None)
    if mismatch is not None:
        print("First mismatch:", mismatch)

    source, target = sample_pair
    print(f"{source} -> {target} path from our code:", reconstruct_path(source, target, ours["next"], labels))
    print(f"{source} -> {target} path from NetworkX:", nx.reconstruct_path(source, target, nx_pred))


verify_against_networkx(matrix1, labels1, ("A", "D"))
print("-" * 60)
verify_against_networkx(matrix2, labels2, ("S", "D"))


## When Should You Use Floyd-Warshall?

| Problem setting | Good first choice | Why |
| --- | --- | --- |
| Single source, non-negative weights | Dijkstra | Usually faster than APSP methods when only one source matters |
| Single source, one target, good heuristic | A* | Often explores much less of the graph |
| Single source, negative edges may exist | Bellman-Ford | Handles negative edges and detects negative cycles reachable from the source |
| All pairs, dense graph, dynamic-programming view | Floyd-Warshall | Simple, elegant, and gives the whole distance matrix directly |
| All pairs, sparse graph, possibly negative edges but no negative cycles | Johnson | Often better than Floyd-Warshall on sparse graphs |

Rule of thumb:

* Floyd-Warshall is most attractive when `V` is moderate and we really want the whole `V x V` distance table
* its matrix form is also excellent for teaching dynamic programming and for visualizing progress after each intermediate vertex


## Verified References for Further Study

Verified on **April 24, 2026**.

* Robert W. Floyd, original paper: [https://doi.org/10.1145/367766.368168](https://doi.org/10.1145/367766.368168)
* Stephen Warshall, original paper: [https://doi.org/10.1145/321105.321107](https://doi.org/10.1145/321105.321107)
* Donald B. Johnson, sparse-graph APSP paper: [https://doi.org/10.1145/321992.321993](https://doi.org/10.1145/321992.321993)
* CLRS, official MIT Press page for *Introduction to Algorithms*, 4th edition: [https://mitpress.mit.edu/9780262046305/introduction-to-algorithms/](https://mitpress.mit.edu/9780262046305/introduction-to-algorithms/)
* NetworkX Floyd-Warshall predecessor and distance documentation: [https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.dense.floyd_warshall_predecessor_and_distance.html](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.dense.floyd_warshall_predecessor_and_distance.html)
* NetworkX Floyd-Warshall NumPy documentation: [https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.dense.floyd_warshall_numpy.html](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.dense.floyd_warshall_numpy.html)
* NetworkX Johnson documentation: [https://networkx.org/documentation/latest/reference/algorithms/shortest_paths/generated/networkx.algorithms.shortest_paths.weighted.johnson.html](https://networkx.org/documentation/latest/reference/algorithms/shortest_paths/generated/networkx.algorithms.shortest_paths.weighted.johnson.html)


## Beyond This Course: Other Shortest-Path Algorithms Not Covered Here

* **Johnson's algorithm**: an APSP algorithm designed for sparse graphs; it combines Bellman-Ford with repeated Dijkstra.
* **Shortest paths in DAGs**: if the graph is a directed acyclic graph, topological order gives a very efficient solution.
* **Bidirectional search**: for some source-target problems, searching forward from the source and backward from the target can reduce work.
* **Yen's algorithm for k-shortest simple paths**: useful when one shortest path is not enough and we want several alternatives.
* **Contraction hierarchies and route-planning methods**: important in large road-network systems where preprocessing is worth the cost.

So the main takeaway is not that Floyd-Warshall solves every shortest-path problem, but that it gives one very clean dynamic-programming solution to the **all-pairs** version of the problem.
